# 12장. 결정트리와 랜덤 포레스트

자료를 나누는 기준을 모델이 스스로 찾으면 무엇을 얻고 무엇을 잃는가.

**준비:** 1장의 전력수요·기온 CSV 다섯 개, 2장의 서울 아파트 실거래가 CSV 네 개, 6장의 경기도 「일반음식점(중국식) 현황」 CSV를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 5분쯤 걸린다.

## 12.1 결정트리: 분할 기준과 깊이

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

def best_cut(x, y, min_leaf=1):
    """x를 어느 값에서 둘로 나누면 두 쪽의 제곱 오차 합이 가장 작아지는지 찾는다. (남는 제곱 오차 합, 나누는 값)을 돌려준다."""
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    n = len(ys)
    run, run2 = np.cumsum(ys), np.cumsum(ys ** 2)
    best = (np.inf, None)
    for i in range(min_leaf, n - min_leaf + 1):          # 왼쪽에 i줄, 오른쪽에 n - i줄
        if i == n or xs[i] == xs[i - 1]:
            continue
        left = run2[i - 1] - run[i - 1] ** 2 / i         # 왼쪽 줄들이 왼쪽 평균에서 벗어난 제곱의 합
        right = (run2[-1] - run2[i - 1]) - (run[-1] - run[i - 1]) ** 2 / (n - i)
        if left + right < best[0]:
            best = (left + right, (xs[i - 1] + xs[i]) / 2)
    return best

before = float(((Y23 - Y23.mean()) ** 2).sum())
after, where = best_cut(T23, Y23)
cold = T23 < where
print(round(before), round(after), round(float(where), 2), round(1 - after / before, 3))
print(int(cold.sum()), round(float(Y23[cold].mean()), 2), int((~cold).sum()), round(float(Y23[~cold].mean()), 2))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 12-1
order = np.argsort(T23)
ts, ys = T23[order], Y23[order]
n = len(ys)
run, run2 = np.cumsum(ys), np.cumsum(ys ** 2)
i = np.arange(1, n)
left = run2[i - 1] - run[i - 1] ** 2 / i
right = (run2[-1] - run2[i - 1]) - (run[-1] - run[i - 1]) ** 2 / (n - i)
mids = (ts[i - 1] + ts[i]) / 2
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(mids, left + right, color=NAVY, lw=1.6)
ax.axhline(before, color=GRAY, lw=1, ls=(0, (4, 2)))
ax.text(-9, before + 35, f"나누기 전 {before:,.0f}", fontsize=8, color="#5b6570", ha="left", va="bottom")
ax.scatter([where], [after], s=42, color=BRICK, zorder=5)
ax.annotate(f"{where:.2f}도에서 나누면 {after:,.0f}로 가장 작다", (where, after), (9.5, 6650), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.set_xlim(-9.5, 31); ax.set_ylim(6200, 9200)
ax.set_xlabel("나누는 기온(°C)", fontsize=8)
ax.set_ylabel("나눈 뒤 두 쪽의 제곱 오차 합", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_12_1.png", dpi=300)
plt.show()

In [ ]:
def grow(X, y, depth, min_leaf=1):
    """특성마다 best_cut을 구해 가장 좋은 특성과 값으로 나눈 뒤, 나뉜 두 쪽에서 같은 일을 반복한다."""
    if depth == 0 or len(y) < 2 * min_leaf:
        return float(y.mean())                           # 잎 노드: 여기에 모인 줄들의 평균이 예측이다
    cuts = [best_cut(X[:, j], y, min_leaf) for j in range(X.shape[1])]
    j = int(np.argmin([c[0] for c in cuts]))
    if cuts[j][1] is None:
        return float(y.mean())
    left = X[:, j] < cuts[j][1]
    return (j, float(cuts[j][1]), grow(X[left], y[left], depth - 1, min_leaf), grow(X[~left], y[~left], depth - 1, min_leaf))

def ask(tree, X):
    """줄마다 뿌리 노드에서 시작해서 조건을 따라 내려가 도착한 잎 노드의 값을 돌려준다."""
    out = np.empty(len(X))
    for i, row in enumerate(X):
        node = tree
        while isinstance(node, tuple):
            node = node[2] if row[node[0]] < node[1] else node[3]
        out[i] = node
    return out

def leaves(tree):
    return 1 if not isinstance(tree, tuple) else leaves(tree[2]) + leaves(tree[3])

X23, X25 = T23[:, None], T25[:, None]                # 특성이 하나뿐인 행렬
two = grow(X23, Y23, 2)
print(two)
print(round(rmse(Y23, ask(two, X23)), 2), round(rmse(Y25, ask(two, X25)), 2))
print([int(((T23 >= lo) & (T23 < hi)).sum()) for lo, hi in [(-99, two[2][1]), (two[2][1], two[1]), (two[1], two[3][1]), (two[3][1], 99)]])

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, 20]:
    tree = grow(X23, Y23, depth)
    rows.append((depth, leaves(tree), round(rmse(Y23, ask(tree, X23)), 2), round(rmse(Y25, ask(tree, X25)), 2)))
by_depth = pd.DataFrame(rows, columns=["깊이", "잎 수", "2023년", "2025년"]).set_index("깊이")
print(by_depth.to_string())

In [ ]:
# 그림 12-3
xs = np.linspace(-9, 30.5, 800)
fig, axes = plt.subplots(3, 1, figsize=(7, 8.2), sharex=True)
for ax, depth in zip(axes, [2, 4, 20]):
    ax.scatter(T23, Y23, s=9, color=to_rgba(GRAY, 0.7), edgecolor="none")
    ax.plot(xs, ask(grow(X23, Y23, depth), xs[:, None]), color=BRICK, lw=1.4)
    row = by_depth.loc[depth]
    frame(ax, f"깊이 {depth}: 잎 {int(row['잎 수'])}개, RMSE 2023년 {row['2023년']:.2f}GW, 2025년 {row['2025년']:.2f}GW")
    ax.set_ylabel("일평균 전력수요(GW)", fontsize=8)
    ax.set_ylim(54, 90)
axes[2].set_xlabel("일평균 기온(°C)", fontsize=8)
[a_.tick_params(labelbottom=True) for a_ in fig.axes]
fig.tight_layout(h_pad=1.4)
fig.savefig("fig_12_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 12-4
fig, ax = plt.subplots(figsize=(7, 3.9))
pos = np.arange(len(by_depth))
ax.plot(pos, by_depth["2025년"], color=BRICK, lw=1.8, marker="o", ms=5, label="2025년(시험 자료)")
ax.plot(pos, by_depth["2023년"], color=NAVY, lw=1.8, marker="s", ms=5, label="2023년(훈련 자료)")
best = int(by_depth["2025년"].idxmin())
ax.annotate(f"깊이 {best}에서 {by_depth['2025년'].min():.2f}GW로 가장 작다", (list(by_depth.index).index(best), by_depth["2025년"].min()), (3.6, 5.3), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.set_xticks(pos, [f"{d}\n(잎 {l})" for d, l in zip(by_depth.index, by_depth["잎 수"])])
ax.set_ylim(-0.3, 6.8)
ax.set_xlabel("트리의 깊이(괄호 안은 잎 노드의 수)", fontsize=8)
ax.set_ylabel("RMSE(GW)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_12_4.png", dpi=300)
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor

theirs = DecisionTreeRegressor(max_depth=2).fit(X23, Y23)
print(theirs.tree_.threshold[theirs.tree_.threshold > -2].round(2).tolist(), round(rmse(Y25, theirs.predict(X25)), 2))

## 12.2 깊은 트리의 과적합과 불안정성

In [ ]:
import os, re

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트를 돌려준다."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)

v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["동이름"] = v["구"] + " " + v["동네"]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["연"] = v["날짜"].dt.year
v["연식"] = v["연"] - v["건축년도"]
v["달"] = (v["연"] - 2023) * 12 + v["날짜"].dt.month          # 2023년 1월이 1, 2025년 12월이 36
v = v.sort_values("날짜").reset_index(drop=True)
print(len(files), raw.shape, len(v))

In [ ]:
a23, a24, a25 = v[v["연"] == 2023], v[v["연"] == 2024], v[v["연"] == 2025]
both = v[v["연"] <= 2024]

def off(actual_log, predicted_log):
    """오차율의 중앙값(%). 2장의 off와 같은 지표를 로그 가격에서 계산한다."""
    return round(float(np.median(np.abs(np.exp(predicted_log - actual_log) - 1)) * 100), 1)

def log_rmse(actual_log, predicted_log):
    return round(float(np.sqrt(((predicted_log - actual_log) ** 2).mean())), 4)

def table_of(df, ref, month=False):
    """면적, 층, 건축년도는 숫자 그대로 쓴다. 동과 단지는 ref에서 가나다순으로 매긴 번호로 바꾼다. 목록에 없는 이름은 -1이 된다."""
    dongs, complexes = pd.Index(sorted(ref["동이름"].unique())), pd.Index(sorted(ref["단지"].unique()))
    cols = [df["면적"], df["층"], df["건축년도"], dongs.get_indexer(df["동이름"]), complexes.get_indexer(df["단지"])]
    return np.column_stack(cols + ([df["달"]] if month else []))

A23, A24 = table_of(a23, a23), table_of(a24, a23)
p23, p24 = np.log(a23["억"].to_numpy()), np.log(a24["억"].to_numpy())
rows = []
for depth in [2, 4, 8, 12, 16, 20, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(A23, p23)
    rows.append((depth or "제한 없음", m.get_n_leaves(), off(p23, m.predict(A23)), off(p24, m.predict(A24)), log_rmse(p24, m.predict(A24))))
print(A23.shape)
print(pd.DataFrame(rows, columns=["깊이", "잎 수", "2023년 %", "2024년 %", "2024년 로그 RMSE"]).set_index("깊이").to_string())

In [ ]:
def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

train = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03"]], ignore_index=True)
test = snapshot("2023-04-03").reset_index(drop=True)
y = test["폐업함"].to_numpy()

print(train.groupby("기준일")["폐업함"].agg(["size", "mean"]).round(3).to_string())
print(len(test), int(y.sum()), round(float(y.mean()), 3))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    """업력 구간 가변수 7개, 시군 가변수 30개, 같은 지번의 폐업 이력, 주변의 영업 중·최근 폐업 가게 수(로그), 좌표 없음 표시."""
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1)))

X, Xt, ytr = columns(train), columns(test), train["폐업함"].to_numpy()
score = Xt @ np.linalg.lstsq(X, ytr, rcond=None)[0]
print(X.shape, round(auc(y, score), 3))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def brier(actual, prob):
    return float(((prob - actual) ** 2).mean())

def seven(df):
    """6장의 특성을 가변수로 바꾸지 않고 숫자 그대로 준다. 업력, 위도, 경도, 지번폐업, 주변, 주변폐업, 시군 번호."""
    return np.column_stack([df["업력"], df["위도"].fillna(0), df["경도"].fillna(0), df["지번폐업"],
                            df["주변"].fillna(-1), df["주변폐업"].fillna(-1), pd.Index(SIGUN).get_indexer(df["시군"])])

year = train["기준일"].dt.year.to_numpy()
fit_on, tune_on = year < 2020, year == 2020
R, Rt = seven(train), seven(test)
rows = []
for depth in [1, 3, 5, 8, 12, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(R[fit_on], ytr[fit_on])
    said = m.predict_proba(R[tune_on])[:, 1]
    rows.append((depth or "제한 없음", m.get_n_leaves(), round(auc(ytr[fit_on], m.predict_proba(R[fit_on])[:, 1]), 3),
                 round(auc(ytr[tune_on], said), 3), round(brier(ytr[tune_on], said), 4), round(float(((said == 0) | (said == 1)).mean()), 3)))
shop_depth = pd.DataFrame(rows, columns=["깊이", "잎 수", "훈련 AUC", "2020년 AUC", "2020년 브라이어", "0 또는 1로 예측한 비율"]).set_index("깊이")
print(shop_depth.to_string())
print(round(brier(ytr[tune_on], ytr[fit_on].mean()), 4))
by_var = DecisionTreeRegressor(max_depth=3, random_state=0).fit(R[fit_on], ytr[fit_on]).predict(R[tune_on])
by_gini = DecisionTreeClassifier(max_depth=3, random_state=0).fit(R[fit_on], ytr[fit_on]).predict_proba(R[tune_on])[:, 1]
print(float(np.abs(by_var - by_gini).max()))

In [ ]:
# 그림 12-5
fig, ax = plt.subplots(figsize=(7, 3.9))
pos = np.arange(len(shop_depth))
ax.plot(pos, shop_depth["훈련 AUC"], color=NAVY, lw=1.8, marker="s", ms=5, label="훈련 자료(2011~2017년 기준일)")
ax.plot(pos, shop_depth["2020년 AUC"], color=BRICK, lw=1.8, marker="o", ms=5, label="검증 자료(2020년 기준일)")
ax.axhline(0.5, color=GRAY, lw=1, ls=(0, (4, 2)))
for i, val in enumerate(shop_depth["2020년 AUC"]):
    ax.text(i, 0.445, f"{val:.3f}", fontsize=8, color=BRICK, ha="center", va="center")
ax.set_xticks(pos, [f"{d}\n(잎 {l:,})" for d, l in zip(shop_depth.index, shop_depth["잎 수"])])
ax.set_ylim(0.4, 1.05)
ax.set_xlabel("트리의 깊이(괄호 안은 잎 노드의 수)", fontsize=8)
ax.set_ylabel("AUC", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_12_5.png", dpi=300)
plt.show()

In [ ]:
rng = np.random.default_rng(0)
firsts = []
for _ in range(200):
    again = rng.integers(0, len(Y23), len(Y23))          # 247일에서 247일을 복원 추출한다
    firsts.append(best_cut(T23[again], Y23[again])[1])
firsts = np.array(firsts)
print(np.quantile(firsts, [0, 0.25, 0.5, 0.75, 1]).round(1).tolist(), int((firsts < 10).sum()), int((firsts > 20).sum()))
for _ in range(4):
    again = rng.integers(0, len(Y23), len(Y23))
    t = grow(X23[again], Y23[again], 2)
    print(round(t[1], 1), [round(side[1], 1) if isinstance(side, tuple) else "잎" for side in (t[2], t[3])], round(rmse(Y25, ask(t, X25)), 2))

In [ ]:
# 그림 12-6
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.hist(firsts, bins=np.arange(0, 27, 1), color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.9)
ax.axvline(where, color=BRICK, lw=1.4, ls=(0, (4, 2)))
top = ax.get_ylim()[1]
ax.text(9.4, top * 0.93, f"점선: 원래 자료의 첫 분할 {where:.2f}도", fontsize=8, color=BRICK)
ax.text(11, top * 0.55, f"10도 아래: {int((firsts < 10).sum())}번", fontsize=8, color=NAVY)
ax.text(19.5, top * 0.3, f"20도 위: {int((firsts > 20).sum())}번", fontsize=8, color=NAVY)
ax.set_xlim(0, 27)
ax.set_xlabel("복원 추출한 자료에서 트리가 고른 첫 분할의 기온(°C)", fontsize=8)
ax.set_ylabel("횟수(200번 가운데)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_12_6.png", dpi=300)
plt.show()

## 12.3 배깅과 랜덤 포레스트

In [ ]:
def bagged(min_leaf, count=200):
    """복원 추출한 자료로 트리를 count개 만든다. 트리마다 2025년에 대한 예측과, 뽑히지 않은 날에 대한 예측을 모은다."""
    said, out_sum, out_n = np.zeros((count, len(Y25))), np.zeros(len(Y23)), np.zeros(len(Y23))
    for b in range(count):
        again = rng.integers(0, len(Y23), len(Y23))
        tree = grow(X23[again], Y23[again], 30, min_leaf)
        said[b] = ask(tree, X25)
        left_out = np.setdiff1d(np.arange(len(Y23)), again)          # 이 트리를 만들 때 뽑히지 않은 날
        out_sum[left_out] += ask(tree, X23[left_out])
        out_n[left_out] += 1
    return said, out_sum / out_n

curves = {}
for min_leaf in [1, 5]:
    said, out_of_bag = bagged(min_leaf)
    alone = np.mean([rmse(Y25, s) for s in said])
    curves[min_leaf] = [rmse(Y25, said[:k].mean(axis=0)) for k in range(1, 201)]
    print(min_leaf, round(alone, 2), [round(curves[min_leaf][k - 1], 2) for k in (5, 20, 50, 200)], round(rmse(Y23, out_of_bag), 2))
    if min_leaf == 5:
        forest25 = said
print(round(float(np.mean([len(np.unique(rng.integers(0, 247, 247))) for _ in range(1000)]) / 247), 3), round(1 - 1 / np.e, 3))

In [ ]:
# 그림 12-7
fig, axes = plt.subplots(2, 1, figsize=(7, 6.6), sharex=True)
order = np.argsort(T25)
for s in forest25[:15]:
    axes[0].plot(T25[order], s[order], color=to_rgba(NAVY, 0.45), lw=0.7)
frame(axes[0], "복원 추출한 자료로 만든 트리 15개의 예측(잎 노드에 5일 이상)")
axes[1].scatter(T25, Y25, s=9, color=to_rgba(GRAY, 0.7), edgecolor="none")
axes[1].plot(T25[order], forest25.mean(axis=0)[order], color=BRICK, lw=1.6)
frame(axes[1], "트리 200개의 평균(벽돌색)과 2025년의 실제 수요(점)")
for ax in axes:
    ax.set_ylabel("일평균 전력수요(GW)", fontsize=8)
    ax.set_ylim(54, 90)
axes[1].set_xlabel("일평균 기온(°C)", fontsize=8)
[a_.tick_params(labelbottom=True) for a_ in fig.axes]
fig.tight_layout(h_pad=1.4)
fig.savefig("fig_12_7.png", dpi=300)
plt.show()

In [ ]:
# 그림 12-8
fig, ax = plt.subplots(figsize=(7, 3.9))
ks = np.arange(1, 201)
ax.plot(ks, curves[1], color=BRICK, lw=1.6)
ax.plot(ks, curves[5], color=NAVY, lw=1.6)
ax.text(200, curves[1][-1] + 0.05, f"잎 노드에 1일 이상: {curves[1][-1]:.2f}GW", fontsize=8, color=BRICK, ha="right", va="bottom")
ax.text(200, curves[5][-1] - 0.05, f"잎 노드에 5일 이상: {curves[5][-1]:.2f}GW", fontsize=8, color=NAVY, ha="right", va="top")
ax.set_xscale("log"); ax.set_xlim(0.9, 220); ax.set_ylim(3.1, 4.7)
plain_ticks(ax.xaxis)
ax.set_xlabel("평균한 트리의 수(로그 축)", fontsize=8)
ax.set_ylabel("2025년 RMSE(GW)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_12_8.png", dpi=300)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rows = []
for pick in [5, 2, 1]:
    # max_features: 노드를 나눌 때마다 특성 5개 가운데 몇 개만 후보로 볼 것인가
    f = RandomForestRegressor(200, max_features=pick, oob_score=True, n_jobs=-1, random_state=0).fit(A23, p23)
    rows.append((pick, off(p23, f.predict(A23)), off(p23, f.oob_prediction_), off(p24, f.predict(A24)), log_rmse(p24, f.predict(A24))))
    if pick == 2:
        grove = f
print(pd.DataFrame(rows, columns=["후보 특성 수", "2023년 %", "OOB %", "2024년 %", "2024년 로그 RMSE"]).set_index("후보 특성 수").to_string())
each = np.array([t.predict(A24) for t in grove.estimators_[:50]])
sigma = float(np.mean([log_rmse(p24, e) for e in each]))
rho = float(np.corrcoef(each - p24)[np.triu_indices(50, 1)].mean())
print(round(sigma, 4), round(rho, 2), round(float(np.sqrt(rho * sigma ** 2 + (1 - rho) * sigma ** 2 / 200)), 4))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rows, woods = [], {}
for min_leaf in [1, 5, 20, 50, 200, 500]:
    woods[min_leaf] = RandomForestClassifier(300, min_samples_leaf=min_leaf, oob_score=True, n_jobs=-1, random_state=0).fit(R[fit_on], ytr[fit_on])
    w_ = woods[min_leaf]
    said = w_.predict_proba(R[tune_on])[:, 1]
    rows.append((min_leaf, round(auc(ytr[fit_on], w_.predict_proba(R[fit_on])[:, 1]), 3), round(auc(ytr[fit_on], w_.oob_decision_function_[:, 1]), 3),
                 round(auc(ytr[tune_on], said), 3), round(brier(ytr[tune_on], said), 4), round(float(said.min()), 2), round(float(said.max()), 2)))
wood_tab = pd.DataFrame(rows, columns=["잎의 최소 줄 수", "훈련 AUC", "OOB AUC", "2020년 AUC", "2020년 브라이어", "최저", "최고"]).set_index("잎의 최소 줄 수")
print(wood_tab.to_string())

In [ ]:
# 그림 12-9
fig, axes = plt.subplots(2, 1, figsize=(7, 6.4), sharex=True)
pos = np.arange(len(wood_tab))
axes[0].plot(pos, wood_tab["훈련 AUC"], color=GRAY, lw=1.8, marker="^", ms=5, label="훈련 자료")
axes[0].plot(pos, wood_tab["OOB AUC"], color=NAVY, lw=1.8, marker="s", ms=5, label="OOB(뽑히지 않은 줄)")
axes[0].plot(pos, wood_tab["2020년 AUC"], color=BRICK, lw=1.8, marker="o", ms=5, label="검증 자료(2020년)")
axes[0].set_ylim(0.5, 1.03); axes[0].set_ylabel("AUC", fontsize=8)
axes[0].legend(frameon=False, fontsize=8, loc="upper right")
frame(axes[0], "AUC")
flat = brier(ytr[tune_on], ytr[fit_on].mean())
axes[1].plot(pos, wood_tab["2020년 브라이어"], color=BRICK, lw=1.8, marker="o", ms=5)
axes[1].axhline(flat, color=GRAY, lw=1.1, ls=(0, (4, 2)))
axes[1].text(len(pos) - 1, flat + 0.0002, f"기준율로만 예측 {flat:.4f}", fontsize=8, color="#5b6570", ha="right", va="bottom")
axes[1].set_ylim(0.1452, 0.1542); axes[1].set_ylabel("브라이어 점수", fontsize=8)
axes[1].set_xticks(pos, [f"{k}\n({lo:.2f}~{hi:.2f})" for k, lo, hi in zip(wood_tab.index, wood_tab["최저"], wood_tab["최고"])])
axes[1].set_xlabel("잎 노드의 최소 줄 수(괄호 안은 검증 자료에 대한 예측 확률의 범위)", fontsize=8)
frame(axes[1], "2020년 검증 자료의 브라이어 점수")
[a_.tick_params(labelbottom=True) for a_ in fig.axes]
fig.tight_layout(h_pad=1.4)
fig.savefig("fig_12_9.png", dpi=300)
plt.show()

## 12.4 트리 모델의 한계: 외삽과 처음 보는 범주

In [ ]:
from scipy import sparse
from sklearn.linear_model import Ridge

def design(df, ref):
    """10장의 특성 행렬에 계약한 달을 수치 특성 하나로 추가했다. 수치 특성 8개에 동과 단지의 가변수."""
    dongs, complexes = pd.Index(sorted(ref["동이름"].unique())), pd.Index(sorted(ref["단지"].unique()))
    num = np.column_stack([np.log(df["면적"]), df["층"], *((np.clip(df["연식"], 0, 59) // 10 == k).astype(float) for k in range(1, 6)), df["달"]])
    blocks = [sparse.csr_matrix(num)]
    for col, cats in [("동이름", dongs), ("단지", complexes)]:
        where_ = cats.get_indexer(df[col])
        known = where_ >= 0
        blocks.append(sparse.csr_matrix((np.ones(known.sum()), (np.where(known)[0], where_[known])), shape=(len(df), len(cats))))
    return sparse.hstack(blocks).tocsr()

pb, p25 = np.log(both["억"].to_numpy()), np.log(a25["억"].to_numpy())
AB, A25 = table_of(both, both, month=True), table_of(a25, both, month=True)
wood = RandomForestRegressor(200, max_features=2, n_jobs=-1, random_state=0).fit(AB, pb)
line = Ridge(alpha=0.01, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(design(both, both), pb)
no_month = RandomForestRegressor(200, max_features=2, n_jobs=-1, random_state=0).fit(AB[:, :5], pb)
said = {"랜덤 포레스트, 달 없음": no_month.predict(A25[:, :5]), "랜덤 포레스트": wood.predict(A25), "릿지": line.predict(design(a25, both))}

quarter = (a25["날짜"].dt.month.to_numpy() - 1) // 3
rows = []
for name, s in said.items():
    rows.append((name, off(p25, s), log_rmse(p25, s), *[round(float((s - p25)[quarter == q].mean()) * 100, 1) for q in range(4)]))
bias = pd.DataFrame(rows, columns=["모델", "2025년 %", "로그 RMSE", "1분기 평균 오차", "2분기", "3분기", "4분기"]).set_index("모델")
print(bias.to_string())
print(round(float(line.coef_[7]) * 100, 2), int(AB[:, 5].max()), int(A25[:, 5].min()))

In [ ]:
# 그림 12-10
fig, ax = plt.subplots(figsize=(7, 3.9))
pos = np.arange(4)
looks = [("랜덤 포레스트, 달 없음", to_rgba(GRAY, 0.5), "#5b6570"), ("랜덤 포레스트", to_rgba(BRICK, 0.55), BRICK), ("릿지", to_rgba(NAVY, 0.22), NAVY)]
for k, (name, fc, ec) in enumerate(looks):
    vals = bias.loc[name, ["1분기 평균 오차", "2분기", "3분기", "4분기"]].to_numpy(dtype=float)
    ax.bar(pos + (k - 1) * 0.27, vals, width=0.25, color=fc, edgecolor=ec, lw=0.9, label=name)
    for x_, val in zip(pos + (k - 1) * 0.27, vals):
        ax.text(x_, val - 0.35, f"{val:.1f}", fontsize=7.6, color=ec, ha="center", va="top")
ax.axhline(0, color="#44505c", lw=0.8)
ax.set_xticks(pos, ["2025년 1분기", "2분기", "3분기", "4분기"])
ax.set_ylim(-17.5, 1.5)
ax.set_ylabel("예측 - 실제의 평균(로그 가격 × 100, 약 %)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="lower left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_12_10.png", dpi=300)
plt.show()

In [ ]:
one = a25.iloc[[0]]                                      # 2025년의 거래 하나를 골라 계약한 달만 바꿔 가며 예측한다
months = np.arange(1, 37)
moved = pd.concat([one] * len(months)).assign(달=months)
path_wood = np.exp(wood.predict(table_of(moved, both, month=True)))
path_line = np.exp(line.predict(design(moved, both)))
print(path_wood[[0, 5, 11, 17, 23, 24, 29, 35]].round(2).tolist())
print(path_line[[0, 5, 11, 17, 23, 24, 29, 35]].round(2).tolist())
print(one[["구", "면적", "층", "건축년도", "달", "억"]].to_string(index=False))

In [ ]:
# 그림 12-11
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.axvspan(24.5, 36.6, color=to_rgba(BRICK, 0.08), lw=0)
ax.plot(months, path_wood, color=BRICK, lw=1.8, label="랜덤 포레스트")
ax.plot(months, path_line, color=NAVY, lw=1.8, label="릿지")
ax.axvline(24.5, color=GRAY, lw=1, ls=(0, (4, 2)))
ax.text(30.5, 12.62, "훈련 자료에 없는 기간\n(2025년)", fontsize=8, color=BRICK, ha="center", linespacing=1.5)
ax.annotate(f"2025년 1월부터 {path_wood[-1]:.2f}억 원으로 같다", (31, path_wood[-1]), (25.3, 13.75), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.set_xticks([1, 7, 13, 19, 25, 31, 36], ["2023.1", "2023.7", "2024.1", "2024.7", "2025.1", "2025.7", "2025.12"])
ax.set_xlim(0.5, 36.6); ax.set_ylim(12.3, 15.6)
ax.set_xlabel("계약한 달(다른 특성은 그대로 두고 이 값만 바꿨다)", fontsize=8)
ax.set_ylabel("같은 집에 대한 예측 가격(억 원)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_12_11.png", dpi=300)
plt.show()

In [ ]:
new = ~a25["단지"].isin(both["단지"]).to_numpy()
print(int(new.sum()))
for name in ["랜덤 포레스트", "릿지"]:
    print(name, off(p25[new], said[name][new]), off(p25[~new], said[name][~new]), round(float((said[name] - p25)[new].mean()) * 100, 1))
print(int(a25.loc[new, "단지"].nunique()), int((AB[:, 4] == -1).sum()), int((A25[:, 4] == -1).sum()))
only_dong = RandomForestRegressor(200, max_features=2, n_jobs=-1, random_state=0).fit(AB[:, [0, 1, 2, 3, 5]], pb)
print(off(p25[new], only_dong.predict(A25[:, [0, 1, 2, 3, 5]])[new]), sorted(both["단지"].unique())[0])

In [ ]:
from sklearn.linear_model import LogisticRegression

def exam(name, said):
    print(f"{name}: AUC {auc(y, said):.3f} 브라이어 {brier(y, said):.4f}")

print(f"기준율로만 예측: 브라이어 {brier(y, ytr[fit_on].mean()):.4f}")
t_logit = LogisticRegression(C=1e6, max_iter=20000).fit(X[fit_on], ytr[fit_on]).predict_proba(Xt)[:, 1]
t_wood = woods[200].predict_proba(Rt)[:, 1]
exam("로지스틱, 특성 41개", t_logit)
exam("랜덤 포레스트, 기본 설정", woods[1].predict_proba(Rt)[:, 1])
exam("랜덤 포레스트, 잎에 200줄 이상", t_wood)

rng = np.random.default_rng(0)
gaps = []
for _ in range(2000):
    pick = rng.integers(0, len(y), len(y))               # 8.2절의 짝지은 부트스트랩
    gaps.append(auc(y[pick], t_logit[pick]) - auc(y[pick], t_wood[pick]))
print(f"로지스틱 - 랜덤 포레스트(잎에 200줄 이상): AUC 차이 {np.mean(gaps):+.3f} 95% 구간 [{np.quantile(gaps, 0.025) + 0:+.3f}, {np.quantile(gaps, 0.975) + 0:+.3f}]")

## 12.5 특성이 서로 조합되어 작용하는 분류 문제: 시간별 전력수요

In [ ]:
hours = h[["GW", "기온"]].copy()
day = (hours.index - pd.Timedelta(hours=1)).normalize()            # 24시의 수요는 그날의 마지막 한 시간이다
hours["시"] = (hours.index - pd.Timedelta(hours=1)).hour + 1
hours["요일"] = day.dayofweek                                       # 월요일이 0, 일요일이 6
hours["월"] = day.month
hours["쉬는날"] = ((day.dayofweek >= 5) | day.isin(HOLIDAYS)).astype(int)
hours["날"] = day

BUSY = 75                                                           # 그 시간의 수요가 75GW 이상이면 '수요가 많은 시간'
hours["많음"] = (hours["GW"] >= BUSY).astype(int)
h23, h25 = hours[day.year == 2023], hours[day.year == 2025]
print(len(h23), int(h23["많음"].sum()), round(float(h23["많음"].mean()), 3), len(h25), int(h25["많음"].sum()), round(float(h25["많음"].mean()), 3))

band = pd.cut(h23["기온"], [-20, 0, 12, 20, 25, 40], right=False, labels=["0도 미만", "0~12도", "12~20도", "20~25도", "25도 이상"])
when = np.select([h23["쉬는날"] == 1, h23["시"].between(10, 18), h23["시"].between(2, 6)], ["쉬는 날", "평일 10~18시", "평일 2~6시"], "평일 그 밖")
print((h23.groupby([band, when], observed=True)["많음"].mean().unstack() * 100).round(1)[["평일 10~18시", "평일 2~6시", "평일 그 밖", "쉬는 날"]].to_string())
print(h23.groupby([band, when], observed=True)["많음"].size().unstack()[["평일 10~18시", "평일 2~6시", "평일 그 밖", "쉬는 날"]].to_string())

In [ ]:
# 그림 12-12
from matplotlib.colors import LinearSegmentedColormap

edges = np.arange(-10, 36, 5)
fig, axes = plt.subplots(2, 1, figsize=(7, 6.4), sharex=True)
shade = LinearSegmentedColormap.from_list("share", ["#ffffff", "#c9d6e3", NAVY])
for ax, off, name in zip(axes, [0, 1], ["평일", "쉬는 날(주말과 공휴일)"]):
    part = h23[h23["쉬는날"] == off]
    row = np.digitize(part["기온"], edges) - 1
    share = part.groupby([row, part["시"].to_numpy()])["많음"].mean().unstack().reindex(index=range(len(edges) - 1), columns=range(1, 25))
    count = part.groupby([row, part["시"].to_numpy()])["많음"].size().unstack().reindex(index=range(len(edges) - 1), columns=range(1, 25))
    share = share.where(count >= 5)                                  # 5시간 미만인 칸은 비워 둔다
    ax.set_facecolor("#eceff3")
    im = ax.imshow(np.ma.masked_invalid(share.to_numpy()), origin="lower", aspect="auto", cmap=shade, vmin=0, vmax=1, extent=[0.5, 24.5, edges[0], edges[-1]])
    for r in range(share.shape[0]):
        for c in range(24):
            v = share.iat[r, c]
            if v == v and v >= 0.05:
                ax.text(c + 1, edges[r] + 2.5, f"{v * 100:.0f}", ha="center", va="center", fontsize=5.6, color="white" if v > 0.55 else NAVY)
    ax.set_yticks(edges); ax.set_xticks(range(1, 25))
    ax.tick_params(labelsize=7.5, labelbottom=True)
    ax.set_ylabel("기온(°C)", fontsize=8)
    ax.set_title(name, fontsize=9, loc="left", color=NAVY, weight="bold")
    ax.spines[["top", "right"]].set_visible(False)
axes[1].set_xlabel("시(1은 0시부터 1시까지)", fontsize=8)
axes[0].set_xlabel("시", fontsize=8)
bar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
bar.set_label("수요가 많은 시간의 비율", fontsize=8); bar.ax.tick_params(labelsize=7.5)
fig.text(0.07, 0.012, "칸 안의 숫자는 비율(%)이고 5% 미만은 적지 않았다. 회색 칸은 2023년에 그 조건인 시간이 5시간 미만인 곳이다.", fontsize=7.2, color="#5b6570")
fig.savefig("fig_12_12.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def log_loss(actual, prob):
    """11.2절의 로그 손실. 실제로 일어난 결과에 예측한 확률의 로그를 구해 부호를 바꾸고 평균한다."""
    prob = np.clip(prob, 1e-12, 1 - 1e-12)
    return float(-(actual * np.log(prob) + (1 - actual) * np.log(1 - prob)).mean())

def added(df):
    """로지스틱 회귀에 줄 특성. 기온의 꺾은 직선 둘, 쉬는 날 표시, 시의 가변수 23개. 특성마다 효과를 따로 더한다."""
    t = df["기온"].to_numpy()
    return np.column_stack([np.maximum(12 - t, 0), np.maximum(t - 20, 0), df["쉬는날"]] + [(df["시"] == k).astype(float) for k in range(2, 25)])

FIVE_COLS = ["기온", "시", "요일", "월", "쉬는날"]
y_h23, y_h25 = h23["많음"].to_numpy(), h25["많음"].to_numpy()
hour_said = {"기준율": np.full(len(y_h25), y_h23.mean())}
hour_said["로지스틱, 기온만"] = LogisticRegression(C=1e6, max_iter=5000).fit(added(h23)[:, :2], y_h23).predict_proba(added(h25)[:, :2])[:, 1]
hour_said["로지스틱, 기온+시+쉬는 날"] = LogisticRegression(C=1e6, max_iter=5000).fit(added(h23), y_h23).predict_proba(added(h25))[:, 1]
for depth in [3, 6, None]:
    hour_said[f"트리, 깊이 {depth or '제한 없음'}"] = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(h23[FIVE_COLS], y_h23).predict_proba(h25[FIVE_COLS])[:, 1]
for min_leaf in [1, 20]:
    hour_said[f"랜덤 포레스트, 잎에 {min_leaf}줄 이상"] = RandomForestClassifier(300, min_samples_leaf=min_leaf, n_jobs=-1, random_state=0).fit(h23[FIVE_COLS], y_h23).predict_proba(h25[FIVE_COLS])[:, 1]
print(pd.DataFrame({k: [round(auc(y_h25, v), 3) if k != "기준율" else 0.5, round(log_loss(y_h25, v), 4), round(brier(y_h25, v), 4)] for k, v in hour_said.items()}, index=["AUC", "로그 손실", "브라이어"]).T.to_string())

codes = pd.factorize(h25["날"])[0]
size = np.bincount(codes)
gap_by_day = np.bincount(codes, (hour_said["로지스틱, 기온+시+쉬는 날"] - y_h25) ** 2) - np.bincount(codes, (hour_said["랜덤 포레스트, 잎에 1줄 이상"] - y_h25) ** 2)
rng, gaps_h = np.random.default_rng(0), []
for _ in range(2000):
    pick = rng.integers(0, len(size), len(size))                     # 하루 단위로 복원 추출한다(8.2절의 짝지은 부트스트랩)
    gaps_h.append(gap_by_day[pick].sum() / size[pick].sum())
print(f"브라이어 차이(로지스틱 - 랜덤 포레스트): {np.mean(gaps_h):+.4f} 95% 구간 [{np.quantile(gaps_h, 0.025):+.4f}, {np.quantile(gaps_h, 0.975):+.4f}]")

In [ ]:
# 그림 12-13
by_hour = pd.DataFrame({"시": h25["시"].to_numpy(),
                        "로지스틱": (hour_said["로지스틱, 기온+시+쉬는 날"] - y_h25) ** 2,
                        "랜덤 포레스트": (hour_said["랜덤 포레스트, 잎에 1줄 이상"] - y_h25) ** 2}).groupby("시").mean()
print(by_hour.loc[[9, 12, 15, 17, 19, 21, 22, 23]].round(3).T.to_string())
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.axvspan(16.5, 23.5, color=to_rgba(BRICK, 0.07), lw=0)
ax.plot(by_hour.index, by_hour["로지스틱"], color=BRICK, lw=1.8, marker="o", ms=3.5, label="로지스틱 회귀(기온 + 시 + 쉬는 날)")
ax.plot(by_hour.index, by_hour["랜덤 포레스트"], color=NAVY, lw=1.8, marker="o", ms=3.5, label="랜덤 포레스트(기본 설정)")
ax.text(20, 0.132, "17~23시: 로지스틱 회귀의 오차가\n랜덤 포레스트의 두 배 안팎이다", fontsize=8, color=BRICK, ha="center", linespacing=1.5)
ax.set_xticks(range(1, 25)); ax.set_xlim(0.5, 24.5); ax.set_ylim(-0.004, 0.15)
ax.set_xlabel("시(1은 0시부터 1시까지)", fontsize=8)
ax.set_ylabel("2025년의 브라이어 점수(시별 평균)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_12_13.png", dpi=300)
plt.show()